# Scenario 8: Agent Control — Runtime Guardrails for AI Agents

---

## What You'll Learn

1. **Connect** to an Agent Control server and register an agent
2. **Create controls** with 5 evaluator types (regex, list, JSON, SQL, Luna-2)
3. **Protect functions** with the `@control()` decorator
4. **Handle actions** — deny (block), steer (correct), warn, log
5. **Organize controls** into reusable policies

## What is Agent Control?

**Agent Control** is a standalone, open-source runtime guardrails platform for AI agents. It lets you add safety controls to your agent **without modifying agent code** — just decorate your functions with `@control()` and the server evaluates them against your rules.

### The Core Formula

Every control follows this structure:

```
Control = Scope + Selector + Evaluator + Action
           │         │          │          │
           │         │          │          └─ What to do: deny, steer, warn, log
           │         │          └──────────── How to check: regex, list, json, sql, luna2
           │         └───────────────────── What to check: input, output, input.query
           └─────────────────────────────── When to check: pre/post, step type, step name
```

### Key Concepts

| Concept | What It Means |
|---------|---------------|
| **Control** | A protection rule: scope + selector + evaluator + action |
| **Evaluator** | The checker: regex, list, JSON schema, SQL parser, or Luna-2 AI |
| **Action** | What happens on match: `deny` (block), `steer` (correct), `warn`, `log` |
| **Policy** | A named group of controls that can be assigned to multiple agents |
| **@control()** | Decorator that automatically evaluates a function against server-defined controls |
| **ControlViolationError** | Exception raised when a `deny` control matches — hard block |
| **ControlSteerError** | Exception raised when a `steer` control matches — provides correction context |

### Architecture

```
Your Agent App                Agent Control Server (localhost:8000)
┌──────────────┐              ┌──────────────────────┐
│ @control()   │──── HTTP ───→│ Evaluate controls    │
│ decorated fn │              │ (regex/list/json/sql) │
│              │─── result ───│ Return: safe/deny/steer│
└──────────────┘              └──────────────────────┘
```

### Prerequisites

- **Docker** running (for the Agent Control server)
- Start the server: `docker compose -f ../agent-control/docker-compose.yml up -d`
- Dependencies installed via `uv sync`
- `.env` file with `AGENT_CONTROL_URL` (default: http://localhost:8000)

## Step 0: Load Environment Variables

Before interacting with the Agent Control server, we need to load configuration from our `.env` file. The key variable is `AGENT_CONTROL_URL`, which tells the SDK where the server is running.

We also apply `nest_asyncio` because the Agent Control SDK is async-first, and Jupyter notebooks already run an event loop. Without this patch, calling `asyncio.run()` inside a notebook would fail with a "cannot run nested event loop" error.

Your `.env` file should contain:

```
AGENT_CONTROL_URL=http://localhost:8000
```

If `AGENT_CONTROL_URL` is not set, we default to `http://localhost:8000`.

In [ ]:
import sys
import os
import asyncio
from pathlib import Path

import nest_asyncio
from dotenv import load_dotenv

nest_asyncio.apply()

for root in [Path.cwd(), Path.cwd().parent]:
    if str(root) not in sys.path:
        sys.path.insert(0, str(root))

env_candidates = [Path.cwd() / '.env', Path.cwd().parent / '.env']
for candidate in env_candidates:
    if candidate.exists():
        load_dotenv(candidate)
        ENV_FILE = candidate
        break
else:
    raise FileNotFoundError('Could not find .env in the repo root or current directory')

AGENT_CONTROL_URL = os.getenv('AGENT_CONTROL_URL', 'http://localhost:8000')
print(f'Loaded credentials from {ENV_FILE}')
print(f'Agent Control server: {AGENT_CONTROL_URL}')

## Step 0b: Import Libraries

Now we import the Agent Control SDK modules. **This is a completely different product from the Galileo SDK** used in notebooks 1–7. The Galileo SDK is for observability (logging traces). Agent Control is for runtime guardrails (enforcing safety rules).

| Import | Purpose |
|--------|---------|
| `agent_control` | The main module — contains `init()`, `refresh_controls()`, `shutdown()`, and control management functions |
| `AgentControlClient` | Async HTTP client for communicating with the Agent Control server |
| `ControlViolationError` | Exception raised when a `deny` control matches — the function call is blocked |
| `ControlSteerError` | Exception raised when a `steer` control matches — provides corrective guidance |
| `control` | The `@control()` decorator — wraps a function so it is automatically evaluated against server-defined controls |

We also set `AGENT_NAME`, which is the unique identifier for our agent on the server. Think of it like a username — controls and policies are associated with this name.

In [ ]:
import agent_control
from agent_control import (
    AgentControlClient,
    ControlViolationError,
    ControlSteerError,
    control,
)

AGENT_NAME = 'galileo-eval-s8-agent'

## Step 1: Verify Server Health

Before doing anything else, we verify that the Agent Control server is running and healthy. The server exposes a `/health` endpoint that returns its status.

**If this cell fails**, the server is not running. Start it with:

```bash
docker compose -f ../agent-control/docker-compose.yml up -d
```

The server runs on `localhost:8000` by default and provides a FastAPI-based REST API. All control creation, evaluation, and policy management happens through this API. The SDK wraps these HTTP calls for convenience, but you could also use the API directly with `curl` or any HTTP client.

In [ ]:
import httpx

async def check_health():
    async with httpx.AsyncClient() as client:
        resp = await client.get(f'{AGENT_CONTROL_URL}/health')
        resp.raise_for_status()
        print(f'Server health: {resp.json()}')
        return resp.json()

asyncio.run(check_health())

## Step 2: Register Agent

`agent_control.init()` is the entry point for connecting your agent to the server. It does several things:

1. **Registers** your agent with the server using the `agent_name` as a unique identifier
2. **Loads** any existing controls already associated with this agent
3. **Starts** a background task that periodically refreshes the control list (so you can update controls on the server without restarting your agent)

The `agent_description` and `agent_version` are metadata — they help you identify agents in the server's admin UI when you have multiple agents registered.

**Important:** If an agent with this name already exists, `init()` reconnects to it (it does not create a duplicate). This makes it safe to re-run this cell.

In [ ]:
agent_control.init(
    agent_name=AGENT_NAME,
    agent_description='Demo agent for learning Agent Control guardrails',
    agent_version='1.0.0',
    server_url=AGENT_CONTROL_URL,
)
print(f'Agent "{AGENT_NAME}" registered with server')

## Step 3: Create a Regex Control (SSN Detection)

Our first control detects **Social Security Numbers** in function outputs. Let's break down each part of the control definition using our core formula:

```
Control = Scope + Selector + Evaluator + Action
           │         │          │          │
         post       output     regex      deny
         (after     (check     (match     (block the
          fn runs)   return     SSN        call and
                     value)     pattern)   raise error)
```

### What each field means

| Field | Value | Explanation |
|-------|-------|-------------|
| `scope.stages: ['post']` | Post-execution | The control runs **after** the function executes, checking its output |
| `selector.path: 'output'` | Output path | Tells the evaluator to check the function's **return value** |
| `evaluator.name: 'regex'` | Regex evaluator | Uses a regular expression to search for matches |
| `evaluator.config.pattern` | `\b\d{3}-\d{2}-\d{4}\b` | Matches the SSN format: 3 digits, dash, 2 digits, dash, 4 digits |
| `action.decision: 'deny'` | Deny action | If the regex matches, **block** the call and raise `ControlViolationError` |
| `action.message` | Error message | Human-readable reason included in the exception |

### Why regex?

Regex is the fastest evaluator — it runs in microseconds and is perfect for detecting structured patterns like SSNs, credit card numbers, phone numbers, and email addresses. No AI model needed.

In [ ]:
async def create_regex_control():
    async with AgentControlClient(base_url=AGENT_CONTROL_URL) as client:
        result = await agent_control.create_control(
            client,
            name='block-ssn-output',
            data={
                'description': 'Block SSNs in function outputs',
                'enabled': True,
                'execution': 'server',
                'scope': {'stages': ['post']},
                'selector': {'path': 'output'},
                'evaluator': {
                    'name': 'regex',
                    'config': {'pattern': r'\b\d{3}-\d{2}-\d{4}\b'},
                },
                'action': {'decision': 'deny', 'message': 'SSN detected in output'},
            },
        )
        print(f'Created regex control: id={result["control_id"]}')
        return result

ssn_control = asyncio.run(create_regex_control())

## Step 4: Create a List Control (Banned Terms)

The **list evaluator** matches values from a predefined list of terms. It is more readable than regex for simple keyword matching and supports several configuration options:

| Option | Values | Explanation |
|--------|--------|-------------|
| `match_mode` | `exact` or `contains` | `exact` requires the full input to match a list item; `contains` checks if any list item appears anywhere in the input |
| `case_sensitive` | `true` / `false` | Whether matching is case-sensitive |
| `match_on` | `match` or `no_match` | `match` triggers on a hit; `no_match` triggers when none of the list items are found |
| `logic` | `any` or `all` | `any` triggers if at least one list item matches; `all` requires every item to match |

This control blocks dangerous system commands in function **inputs** (pre-execution). It uses `contains` mode with `any` logic, meaning if the input contains **any one** of the banned terms (case-insensitive), the call is blocked before the function even runs.

### Scope: pre vs post

Notice this control uses `stages: ['pre']` instead of `['post']`. Pre-stage controls evaluate the function's **input arguments** before execution. This is important for dangerous operations — you want to block `DROP TABLE` **before** it reaches your database, not after.

In [ ]:
async def create_list_control():
    async with AgentControlClient(base_url=AGENT_CONTROL_URL) as client:
        result = await agent_control.create_control(
            client,
            name='block-dangerous-commands',
            data={
                'description': 'Block dangerous system commands in inputs',
                'enabled': True,
                'execution': 'server',
                'scope': {'stages': ['pre']},
                'selector': {'path': 'input'},
                'evaluator': {
                    'name': 'list',
                    'config': {
                        'values': ['DROP TABLE', 'DELETE FROM', 'TRUNCATE', 'rm -rf', 'sudo'],
                        'case_sensitive': False,
                        'match_mode': 'contains',
                        'match_on': 'match',
                        'logic': 'any',
                    },
                },
                'action': {'decision': 'deny', 'message': 'Dangerous command detected'},
            },
        )
        print(f'Created list control: id={result["control_id"]}')
        return result

banned_control = asyncio.run(create_list_control())

## Step 5: Associate Controls with Agent

Controls and agents exist independently on the server. Creating a control does **not** automatically apply it to any agent. You must explicitly associate them.

This separation is powerful because:

- **One control can protect multiple agents** — create a "block SSN" control once and apply it to every agent in your system
- **Controls can be managed centrally** — a security team can create and update controls without touching agent code
- **Controls can be toggled** — disable a control for one agent without affecting others

After associating, we verify by listing all controls on the server and all controls attached to our specific agent.

In [ ]:
async def associate_controls():
    async with AgentControlClient(base_url=AGENT_CONTROL_URL) as client:
        await agent_control.add_agent_control(client, AGENT_NAME, ssn_control['control_id'])
        await agent_control.add_agent_control(client, AGENT_NAME, banned_control['control_id'])

        # Verify
        result = await agent_control.list_controls(client)
        print(f'Total controls on server: {result.get("total", "?")}')

        agent_controls = await agent_control.agents.list_agent_controls(client, AGENT_NAME)
        print(f'Controls on agent: {len(agent_controls.get("controls", []))}')

asyncio.run(associate_controls())

## Step 6: Protect Functions with @control() — Test Deny

The `@control()` decorator is where everything comes together. It wraps an async function and adds two evaluation checkpoints:

```
customer_lookup("Show me SSN for John")
        │
        ▼
  ┌────────────────────┐
  │ PRE-STAGE evaluation │  ← Check input against pre-stage controls
  │ (banned commands?)   │    If deny → raise ControlViolationError (function never runs)
  └────────────────────┘
        │ pass
        ▼
  ┌────────────────────┐
  │ Function executes    │  ← Your actual function code runs
  │ returns result       │
  └────────────────────┘
        │
        ▼
  ┌────────────────────┐
  │ POST-STAGE evaluation│  ← Check output against post-stage controls
  │ (SSN in output?)     │    If deny → raise ControlViolationError (output suppressed)
  └────────────────────┘
        │ pass
        ▼
  Return result to caller
```

We first call `agent_control.refresh_controls()` to ensure the decorator has the latest control definitions from the server. Then we test two scenarios:

1. **Safe call** — returns normal customer info (no SSN), passes all controls
2. **Blocked call** — returns an SSN in the output, caught by the regex control, raises `ControlViolationError`

In [ ]:
# Refresh controls so the decorator knows about our new controls
agent_control.refresh_controls()

@control()
async def customer_lookup(query: str) -> str:
    """Simulate a customer lookup that might return sensitive data."""
    if 'ssn' in query.lower():
        return 'Customer John Doe, SSN: 123-45-6789'
    return f'Customer info for: {query}'

async def test_controls():
    # Safe call — no SSN in output
    result = await customer_lookup('Show me account balance for C-1234')
    print(f'Safe call result: {result}')

    # Blocked call — SSN in output triggers regex control
    try:
        result = await customer_lookup('Show me SSN for John Doe')
        print(f'Got: {result}')
    except ControlViolationError as e:
        print(f'BLOCKED by control "{e.control_name}": {e.message}')

asyncio.run(test_controls())

## Step 7: Handle ControlViolationError Gracefully

In production, you never want an unhandled exception to crash your agent. The recommended pattern is to wrap every `@control()`-decorated function call in a `try/except` block that catches `ControlViolationError` and returns a safe fallback response.

The exception object contains useful information:

| Attribute | What It Contains |
|-----------|------------------|
| `e.control_name` | The name of the control that triggered (e.g., `block-ssn-output`) |
| `e.message` | The human-readable message defined in the control's action |

This lets you log the violation, return a user-friendly message, and optionally alert your security team — all without the end user seeing a raw error.

In [ ]:
async def safe_agent_call(query: str) -> str:
    """Production pattern: wrap @control() calls with fallback."""
    try:
        return await customer_lookup(query)
    except ControlViolationError as e:
        return f'I cannot process this request. (Reason: {e.control_name})'

# Test both paths
for query in ['What is my balance?', 'Show me SSN for John']:
    result = asyncio.run(safe_agent_call(query))
    print(f'Query: {query!r} \u2192 {result}')

## Step 8: Create a Steer Control + Handle ControlSteerError

Not every control violation should be a hard block. Sometimes you want to **guide** the agent to fix its behavior instead of stopping it entirely. That is what the `steer` action does.

### Deny vs Steer

| Action | Exception | Behavior |
|--------|-----------|----------|
| `deny` | `ControlViolationError` | Hard block — the output is suppressed, the caller gets an error |
| `steer` | `ControlSteerError` | Soft correction — the output is flagged, and the exception includes guidance on how to fix it |

The `ControlSteerError` exception contains:

| Attribute | What It Contains |
|-----------|------------------|
| `e.control_name` | The name of the control that triggered |
| `e.message` | Corrective guidance (e.g., "Please rephrase formally") |
| `e.output_data` | The original output that triggered the control |

In production, you would use this in a **retry loop**: catch the steer error, feed `e.message` back into your LLM prompt as a system instruction, and regenerate the response. This creates a self-correcting agent that improves its output based on your rules.

This control uses the list evaluator to detect informal language ("hey", "gonna", "lol", etc.) and steers the agent to use formal tone.

In [ ]:
async def create_steer_control():
    async with AgentControlClient(base_url=AGENT_CONTROL_URL) as client:
        result = await agent_control.create_control(
            client,
            name='steer-formal-tone',
            data={
                'description': 'Steer responses to use formal tone',
                'enabled': True,
                'execution': 'server',
                'scope': {'stages': ['post']},
                'selector': {'path': 'output'},
                'evaluator': {
                    'name': 'list',
                    'config': {
                        'values': ['hey', 'yo', 'sup', 'gonna', 'wanna', 'lol'],
                        'case_sensitive': False,
                        'match_mode': 'contains',
                        'match_on': 'match',
                        'logic': 'any',
                    },
                },
                'action': {
                    'decision': 'steer',
                    'message': 'Response uses informal language. Please rephrase formally.',
                },
            },
        )
        await agent_control.add_agent_control(client, AGENT_NAME, result['control_id'])
        print(f'Created steer control: id={result["control_id"]}')
        return result

steer_control = asyncio.run(create_steer_control())
agent_control.refresh_controls()

@control()
async def draft_response(query: str) -> str:
    return f'Hey! Gonna help you with: {query}'

async def test_steer():
    try:
        result = await draft_response('refund request')
        print(f'Got: {result}')
    except ControlSteerError as e:
        print(f'STEERED by "{e.control_name}": {e.message}')
        print(f'Original output: {e.output_data}')
        # In production: retry with corrected prompt

asyncio.run(test_steer())

## Step 9: SQL Evaluator (Database Safety)

If your agent generates SQL queries (a common pattern for data-analyst agents), the **SQL evaluator** is essential. It parses queries using an AST parser ([sqlglot](https://github.com/tobymao/sqlglot)) and can enforce rules that are impossible to get right with regex:

| Rule | What It Does |
|------|--------------|
| `allowed_operations: ['SELECT']` | Only allow SELECT — block DROP, DELETE, INSERT, UPDATE, ALTER, etc. |
| `require_limit_on_select: true` | Every SELECT must have a LIMIT clause (prevents full-table scans) |
| `max_limit: 100` | The LIMIT value cannot exceed 100 |

### Why not just use regex?

Regex-based SQL filtering is notoriously fragile. Consider:

- `SELECT * FROM users WHERE name = 'Robert'); DROP TABLE users;--` — SQL injection
- `SELECT * FROM drop_table_log` — false positive (table name contains "DROP TABLE")
- `SELECT name /* DROP TABLE users */ FROM users` — injection hidden in comments

The SQL evaluator parses the query into an AST (abstract syntax tree), so it understands the **structure** of the query, not just the text. It correctly handles all of these cases.

This control uses `scope.stages: ['pre']` because we want to block bad SQL **before** it reaches the database.

In [ ]:
async def create_sql_control():
    async with AgentControlClient(base_url=AGENT_CONTROL_URL) as client:
        result = await agent_control.create_control(
            client,
            name='sql-read-only',
            data={
                'description': 'Restrict SQL to SELECT with LIMIT',
                'enabled': True,
                'execution': 'server',
                'scope': {'stages': ['pre']},
                'selector': {'path': 'input'},
                'evaluator': {
                    'name': 'sql',
                    'config': {
                        'allowed_operations': ['SELECT'],
                        'require_limit_on_select': True,
                        'max_limit': 100,
                    },
                },
                'action': {'decision': 'deny', 'message': 'Only SELECT queries with LIMIT allowed'},
            },
        )
        await agent_control.add_agent_control(client, AGENT_NAME, result['control_id'])
        print(f'Created SQL control: id={result["control_id"]}')
        return result

sql_control = asyncio.run(create_sql_control())
agent_control.refresh_controls()

@control()
async def query_database(query: str) -> str:
    return f'Executed: {query}'

async def test_sql():
    # Safe: SELECT with LIMIT
    try:
        result = await query_database('SELECT name, email FROM users LIMIT 10')
        print(f'Safe SQL: {result}')
    except ControlViolationError as e:
        print(f'Blocked: {e.message}')

    # Blocked: DROP TABLE
    try:
        result = await query_database('DROP TABLE users')
        print(f'Got: {result}')
    except ControlViolationError as e:
        print(f'Blocked: {e.message}')

    # Blocked: SELECT without LIMIT
    try:
        result = await query_database('SELECT * FROM users')
        print(f'Got: {result}')
    except ControlViolationError as e:
        print(f'Blocked: {e.message}')

asyncio.run(test_sql())

## Step 10: JSON Evaluator (Output Validation)

The **JSON evaluator** validates that function outputs match a predefined schema. This is especially useful when your agent produces structured data (API responses, tool call results, etc.) and you need to guarantee the output format is correct.

### Configuration options

| Option | What It Does |
|--------|--------------|
| `required_fields` | List of field names that must be present in the JSON |
| `field_types` | Expected type for each field (`string`, `number`, `boolean`, `array`, `object`) |
| `field_constraints` | Additional constraints per field (e.g., `enum` for allowed values) |

### Why use this?

LLMs are notoriously unreliable at producing consistently structured output. Even with careful prompting, an LLM might:

- Omit required fields
- Use wrong field names (`Status` instead of `status`)
- Return unexpected values (`"success"` instead of `"ok"`)

The JSON evaluator catches these issues at runtime, before the malformed output reaches downstream systems.

This control checks that API responses contain `status` and `data` fields, and that `status` is one of `"ok"` or `"error"`.

In [ ]:
async def create_json_control():
    async with AgentControlClient(base_url=AGENT_CONTROL_URL) as client:
        result = await agent_control.create_control(
            client,
            name='validate-api-response',
            data={
                'description': 'Validate structured API responses',
                'enabled': True,
                'execution': 'server',
                'scope': {'stages': ['post']},
                'selector': {'path': 'output'},
                'evaluator': {
                    'name': 'json',
                    'config': {
                        'required_fields': ['status', 'data'],
                        'field_types': {'status': 'string'},
                        'field_constraints': {
                            'status': {'enum': ['ok', 'error']},
                        },
                    },
                },
                'action': {'decision': 'deny', 'message': 'Response does not match required schema'},
            },
        )
        await agent_control.add_agent_control(client, AGENT_NAME, result['control_id'])
        print(f'Created JSON control: id={result["control_id"]}')
        return result

json_control = asyncio.run(create_json_control())
agent_control.refresh_controls()

import json

@control()
async def get_api_response(endpoint: str) -> str:
    if endpoint == '/valid':
        return json.dumps({'status': 'ok', 'data': {'users': 42}})
    return json.dumps({'result': 'something', 'code': 200})  # missing 'status' and 'data'

async def test_json():
    # Valid response
    try:
        result = await get_api_response('/valid')
        print(f'Valid: {result}')
    except ControlViolationError as e:
        print(f'Blocked: {e.message}')

    # Invalid response (missing required fields)
    try:
        result = await get_api_response('/invalid')
        print(f'Got: {result}')
    except ControlViolationError as e:
        print(f'Blocked: {e.message}')

asyncio.run(test_json())

## Step 11: Luna-2 Evaluator (Optional — Galileo Integration)

**Luna-2** is Galileo's proprietary AI model for content safety. Unlike the previous evaluators (regex, list, JSON, SQL) which use deterministic rules, Luna-2 uses a neural network to **score** content on safety dimensions.

### Available Luna-2 metrics

| Metric | What It Detects |
|--------|-----------------|
| `input_toxicity` | Toxic, offensive, or harmful content in inputs |
| `output_toxicity` | Toxic content in outputs |
| `prompt_injection` | Attempts to override the system prompt |
| `input_pii` | Personally identifiable information in inputs |

### How it works

The evaluator sends the content to Galileo's Luna-2 API, which returns a score between 0.0 and 1.0. You configure a threshold using `operator` and `target_value`:

- `operator: 'gt'`, `target_value: 0.5` → trigger if toxicity score > 0.5
- `operator: 'lt'`, `target_value: 0.3` → trigger if score < 0.3

### Prerequisites

This evaluator requires a `GALILEO_API_KEY` in your `.env` file. If the key is not set, this cell is skipped. This is the one point where Agent Control integrates with the Galileo platform.

In [ ]:
if os.getenv('GALILEO_API_KEY'):
    async def create_luna2_control():
        async with AgentControlClient(base_url=AGENT_CONTROL_URL) as client:
            result = await agent_control.create_control(
                client,
                name='block-toxic-luna2',
                data={
                    'description': 'Block toxic content using Galileo Luna-2',
                    'enabled': True,
                    'execution': 'server',
                    'scope': {'stages': ['pre']},
                    'selector': {'path': 'input'},
                    'evaluator': {
                        'name': 'galileo.luna2',
                        'config': {
                            'metric': 'input_toxicity',
                            'operator': 'gt',
                            'target_value': 0.5,
                        },
                    },
                    'action': {'decision': 'deny', 'message': 'Toxic content detected by Luna-2'},
                },
            )
            await agent_control.add_agent_control(client, AGENT_NAME, result['control_id'])
            print(f'Created Luna-2 control: id={result["control_id"]}')
            return result

    luna2_control = asyncio.run(create_luna2_control())
else:
    print('Skipping Luna-2 evaluator: GALILEO_API_KEY not set in .env')
    print('Set GALILEO_API_KEY to enable AI-powered content safety detection')

## Step 12: Policies — Group and Assign Controls

As your system grows, managing controls one-by-one becomes tedious. **Policies** solve this by grouping controls into named collections that can be assigned to agents as a unit.

### Why use policies?

| Scenario | Without Policies | With Policies |
|----------|-----------------|---------------|
| 10 agents, same 5 controls | 50 individual associations | 1 policy, 10 assignments |
| Add a new control to all agents | Update each agent individually | Add control to policy — all agents get it |
| Different rules for prod vs dev | Manual tracking | `production-safety` policy vs `dev-permissive` policy |

### How it works

```
Policy: "production-safety"
  ├─ block-ssn-output (regex)
  ├─ block-dangerous-commands (list)
  └─ sql-read-only (sql)
       │
       └─── assigned to ───→ agent-1, agent-2, agent-3
```

The workflow is:
1. **Create** a policy with a descriptive name
2. **Add controls** to the policy
3. **Assign** the policy to one or more agents

When the agent calls `refresh_controls()`, it receives all controls from its directly-associated controls **and** from all assigned policies.

In [ ]:
async def setup_policy():
    async with AgentControlClient(base_url=AGENT_CONTROL_URL) as client:
        # Create policy
        policy = await agent_control.policies.create_policy(client, name='production-safety')
        policy_id = policy['policy_id']
        print(f'Created policy: id={policy_id}')

        # Add controls to the policy
        await agent_control.add_control_to_policy(client, policy_id, ssn_control['control_id'])
        await agent_control.add_control_to_policy(client, policy_id, banned_control['control_id'])

        # Assign policy to agent
        await agent_control.add_agent_policy(client, AGENT_NAME, policy_id)

        # Verify
        policy_controls = await agent_control.list_policy_controls(client, policy_id)
        print(f'Policy has {len(policy_controls.get("control_ids", []))} controls')

        return policy

policy = asyncio.run(setup_policy())

## Clean Up

The Agent Control server persists state in its database. This cleanup cell deletes all controls we created and shuts down the SDK's background tasks.

To fully reset the server (delete all agents, controls, and policies), stop the server and remove its volumes:

```bash
docker compose -f ../agent-control/docker-compose.yml down -v
```

In production, you would not delete controls — they would stay on the server permanently, managed by your security team.

In [ ]:
async def cleanup():
    async with AgentControlClient(base_url=AGENT_CONTROL_URL) as client:
        # Delete all controls we created
        for ctrl in [ssn_control, banned_control, steer_control, sql_control, json_control]:
            try:
                await agent_control.delete_control(client, ctrl['control_id'], force=True)
            except Exception:
                pass

    agent_control.shutdown()
    print('Cleaned up controls and shut down agent')
    print('To fully reset: docker compose -f ../agent-control/docker-compose.yml down -v')

asyncio.run(cleanup())